# Continuous-Variable Quantum Transpiler: Schrödinger Cat State Engine
### End-to-End Tutorial: Continuous-Variable Cat State Generation & Non-Classicality Verification

This tutorial simulates the preparation, open quantum dynamics, and microwave pulse transpilation of an error-corrected continuous-variable Schrödinger cat state:
$$\vert \text{cat} \rangle = \frac{1}{\sqrt{2}} (\vert \alpha \rangle + i \vert -\alpha \rangle)$$
with coherent amplitude $\alpha = 2.0$.

In [ ]:
import sys
import os
import numpy as np

# Ensure project root is on path
sys.path.insert(0, os.path.abspath(".."))

from solver.lindblad import LindbladSolver, cat_state, coherent_state, ket2dm
from solver.wigner import WignerCalculator, compute_wigner
from optimizer.noise_models import QuantumNoiseModel, CavityNoiseParameters
from transpiler.passes import CVCircuit, DecompositionPass, WaveformSynthesisPass
from transpiler.qiskit_plugin import QiskitPulseBridge

fock_dim = 30
alpha = 2.0
print(f"Target State: Cat State |cat> (alpha={alpha}, Fock dim={fock_dim})")

## 1. Yurke-Stoler Non-Linear Kerr Evolution & Master Equation Solver
Simulating Kerr non-linearity $\hat{H}_{\text{Kerr}} = -K (\hat{a}^\dagger \hat{a})^2$ under photon loss and dephasing.

In [ ]:
solver = LindbladSolver(fock_dim=fock_dim)
noise_params = CavityNoiseParameters(t1_us=200.0, t_phi_us=400.0, thermal_photons=0.002)
noise_model = QuantumNoiseModel(noise_params)
c_ops = noise_model.get_c_ops(fock_dim)

psi0 = coherent_state(alpha, fock_dim=fock_dim)
rho0 = ket2dm(psi0)

kerr_mhz = 0.5
H_kerr = -kerr_mhz * (solver.n_op @ solver.n_op)
t_cat_us = np.pi / (2.0 * kerr_mhz)

print(f"Simulating Kerr evolution to t_cat = {t_cat_us:.3f} us...")
t_span = np.linspace(0.0, t_cat_us, 30)
res = solver.solve(rho0, t_span, H_kerr, c_ops, substeps=10)
final_rho = res.final_state

# Ideal target state for fidelity benchmarking
U_ideal = np.diag(np.exp(-1j * (-kerr_mhz * (np.arange(fock_dim) ** 2)) * t_cat_us))
target_psi = U_ideal @ psi0
target_psi = target_psi / np.linalg.norm(target_psi)

fidelity = solver.state_fidelity(final_rho, target_psi)
purity = solver.purity(final_rho)
avg_photons = float(np.real(np.trace(final_rho @ solver.n_op)))

print(f"Cat State Fidelity: {fidelity * 100:.2f}%")
print(f"Purity Tr(rho^2): {purity:.4f}")
print(f"Photon Number <n>: {avg_photons:.2f}")

## 2. 2D Phase-Space Wigner Quasiprobability Distribution & Negativity
Visualizing quantum interference fringes and computing non-classical negativity volume $\delta(\hat{\rho}) = \iint (|W(x, p)| - W(x, p)) dx dp$.

In [ ]:
xvec, pvec, W = compute_wigner(final_rho, x_range=4.0, num_points=100)
calc = WignerCalculator(xvec, pvec)
neg_vol = calc.negative_volume(W)
total_prob = calc.total_probability(W)

print(f"Wigner Total Probability: {total_prob:.4f}")
print(f"Non-Classicality Negativity Volume: {neg_vol:.4f}")

## 3. Physical Microwave Pulse Transpilation & Qiskit Schedule Synthesis
Compiling continuous-variable quantum gates into calibrated physical microwave pulse waveforms.

In [ ]:
circuit = CVCircuit(num_modes=1)
circuit.displacement(alpha=complex(alpha, 0.0), mode=0)
circuit.kerr(chi=np.pi / 2, mode=0)

decomposer = DecompositionPass()
pulses = decomposer.run(circuit)
synthesizer = WaveformSynthesisPass(sampling_rate_gsa_s=2.0)
waveforms = synthesizer.synthesize(pulses)

bridge = QiskitPulseBridge()
qiskit_schedule = bridge.transpile_to_qiskit_schedule(circuit)

print(f"Compiled Pulses: {len(pulses)}")
for p in pulses:
    print(f"  Channel: {p.channel} | Duration: {p.duration_ns} ns | Amplitude: {p.amplitude:.3f}")
print("\nQiskit Schedule Dictionary Structure:")
print(qiskit_schedule)